# Track Data v2: Thu thap Da Chinh sach + Kiem dinh Chat luong Chuan SOTA (Model-Neutral)

Notebook nay chay thuan tren **CPU** (tren may ca nhan hoac Kaggle CPU) de tao bo du lieu v2 giai quyet 5 diem yeu cua Wave 0:
1. **Da hanh vi (Multi-Policy Coverage - arXiv:2408.07986):** `EmptyDayConsistentPolicy` (Anchor) + `Orthogonal Single-Axis Explore` + `PIDWaterResetPolicy`.
2. **Kich thich doc lap (`VIF` check):** Chi nhich 1 truc hanh dong moi block (`30 phut`) thay vi giat cung luc ca 4 truc.
3. **POMDP & Hinh hoc day du:** Bo sung `ep_id`, `step_idx`, `dow_sin/cos`, `zone_ua_norm`, `dT_30m`, `d_oat_30m`, `temp_spread_ahu`.
4. **Chu ky 7 ngay day du (`STEPS_COLLECT = 2016`):** Bao phu ca Thu Tu -> Thu Sau va chuyen giao cuoi tuan (`0-864` vs `864-2016`).
5. **Phan ra BOPTEST KPI:** Luu rieng `comfort_regret`, `energy_cost`, `carbon_emission` va tu xuat bao cao audit (`data_audit_v2_report.png`).


In [ ]:
# Cell 1: Cau hinh & Kiem tra Moi truong (Local CPU hoac Kaggle CPU)
import os, sys, json
from pathlib import Path
import numpy as np

# --- CAU HINH DATA V2 ---
STEPS_COLLECT = 2016       # 7 ngay day du (Sun -> Sat). Co the ha xuong 864 neu muon test nhanh, hoac 8640 cho full 30 ngay
MAX_SCENARIOS = None       # Dat so nguyen (vd: 6 = 1 kich ban/toa) de smoke test nhanh; None = full 30 kich ban
INCLUDE_PID_WATER = True   # Thu them luot PIDWaterResetPolicy de mo rong State-Action Coverage
BLOCK_STEPS = 6            # 30 phut giu kich thich don truc
N_STEP = 6
GAMMA = 0.96
SEED = 42

candidates = [
    Path('/kaggle/input/competitions/neurips-2026-smart-buildings'),
    Path('/kaggle/input/neurips-2026-smart-buildings'),
]
DATA_ROOT = next((p for p in candidates if (p / 'smart_buildings').exists()), None)
if DATA_ROOT is None:
    for base in [Path.cwd()] + list(Path.cwd().parents):
        if (base / 'smart_buildings').exists():
            DATA_ROOT = base
            break
assert DATA_ROOT is not None, 'Khong tim thay smart_buildings package!'

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else (DATA_ROOT / 'data_v2_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [OUT_DIR, Path.cwd(), DATA_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.chdir(DATA_ROOT)
from smart_buildings.evaluate import load_hybrid_environment, create_building_meta_data
from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy
from collect_data_v2 import run_collect_and_audit_v2

TRAIN_DIR = DATA_ROOT / 'smart_buildings' / 'training_scenarios'
ALL_SCENARIOS = sorted([p.parent for p in TRAIN_DIR.glob('*/*/config.gin')])
if MAX_SCENARIOS is not None:
    ALL_SCENARIOS = ALL_SCENARIOS[:int(MAX_SCENARIOS)]

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'OUT_DIR   : {OUT_DIR}')
print(f'Kich ban  : {len(ALL_SCENARIOS)} | Steps: {STEPS_COLLECT} | PID_WATER: {INCLUDE_PID_WATER}')


In [ ]:
# Cell 2: Chay Thu thap v2 & In Bao cao Kiem dinh Chat luong Data (SOTA Audit)
BUF_v2, df_audit_v2, df_th_v2, anchor_scores_v2, audit_summary = run_collect_and_audit_v2(
    ALL_SCENARIOS,
    load_hybrid_environment,
    create_building_meta_data,
    pid_cls=PIDWaterResetPolicy if INCLUDE_PID_WATER else None,
    out_dir=OUT_DIR,
    steps=STEPS_COLLECT,
    n_step=N_STEP,
    gamma=GAMMA,
    block_steps=BLOCK_STEPS,
    include_pid_water=INCLUDE_PID_WATER,
    seed=SEED,
)

print('\nCac file nhe co the copy vao experiment_logs/wave_data_v2/ de push GitHub:')
for f in sorted(OUT_DIR.glob('*.json')) + sorted(OUT_DIR.glob('*.png')):
    print(' -', f)
